# Plot Back-slip inversion of the GNSS displacement measurements of interseismic locking (coupling) at Nicoya (Costa Rica) within a homogeneous half-space



In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils as ut

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"

# define mesh name
# meshname = "nicoya"  # larger fault interface
meshname = "nicoya2"   # This has a smaller fault interface

# define the type of disp. data to use
type = "both"   # use both trench parallel and normal components
# type = "dip"    # use only trench normal components     

# Read the plate interface file
plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
df_plate = ut.parse_plate_interface_file(datadir + plate_file)
depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the plate file in GMT grd format
grd_file = "/home/staff/chao/SSEinv/Nicoya/plateinterface/cam_slab2_dep_02.24.18.grd"

# Read the trench file
trench_file = "plateinterface/trench.gmt.inuse"

# read in the lon and lat of stations
# obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

# note that the height is in m, Dt in years, the original displacement data is in mm/yr
# the disp relative to the stable Caribbean plate will be used in the inversion
# From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                 names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                        'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                        'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())


In [ ]:
# Decide the weights of the horizontal, vertical components
f_h, f_v = 1, 1/2
# Print the weights of the data
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
# Take the inverse for saving the name of the weights
w_h, w_v = int(1/f_h), int(1/f_v)

# the optimal weight combination comes from the L-curve test
# rho_s = 1e7
# rho_s = 1e8
rho_s = 1e9

# gamma_val_H1 = 1e2
gamm
a_val_H1 = 4e2
# gamma_val_H1 = 1e3

delta_val_L2 = gamma_val_H1 / rho_s 

# file identifier
# inv_str = f"_locking_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"

inv_str = f"_locking{type}_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"

# displacement noise standard deviations
error_e, error_n, error_v = df['vx_std_Car']/1e3, df['vy_std_Car']/1e3, df['vz_std_Car']/1e3

print(inv_str)
# print(error_e)

# flag to indicate whether to save figures
# flag_savefig = True
flag_savefig = False



In [ ]:
# Load the observed surface displacement, all in meters
outFileName = 'd_obs_' + meshname + inv_str + '.txt'
u_obs = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                     names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
u_obs.head()

# Load the predicted surface displacement, all in meters
outFileName = 'd_cal_' + meshname + inv_str + '.txt'
u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                     names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
print(outFileName)
u_pred.head()

u_res = u_pred.copy()
u_res['ux'] = u_obs['ux'] - u_pred['ux']
u_res['uy'] = u_obs['uy'] - u_pred['uy']
u_res['uz'] = u_obs['uz'] - u_pred['uz']
u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
u_res.head()


In [ ]:
# Load the fault geometry, all in meters
outFileName = 'fault_geometry_' + meshname + '.txt'
loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                    names=['xf', 'yf', 'zf'])
loc_f.head()

In [ ]:
# Load the inferred slip on the fault, all in meters
outFileName = 'm_s_fault_' + meshname + inv_str + '.txt'

if type == 'both':
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_strike', 's_dip'])

    cols_to_convert = ['s_strike', 's_dip']
    m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm

    m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)
    print(m_s['s_strike'].min(), m_s['s_strike'].max())
    print(m_s['s_dip'].min(), m_s['s_dip'].max())
    print(m_s['mag'].max())

elif type == 'dip':
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_dip'])
    cols_to_convert = ['s_dip']
    m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm

    m_s['mag'] = m_s['s_dip']
    print(m_s['s_dip'].min(), m_s['s_dip'].max())
    print(m_s['mag'].max())

In [ ]:
# define interseismic coupling between the two plates as the ratio of back normal slip rate
# (m_est vector) to local trench-normal convergence rate (m_est/V_norm.)
V_norm = 78.5   # trench-normal of Cocos-Caribbean motion, mm/yr
V_const_para = 11     # only remove the a constant value from trench parallel component, mm/yr 
V_para = 27-V_const_para    # trench-parallel of Cocos-Caribbean motion, mm/yr       
m_s['locking'] = m_s['s_dip']/V_norm
# m_s['locking'] = m_s['mag']/V_norm
print(m_s['locking'].min(), m_s['locking'].max())

#Do we need to scale it so that the max. is 1?
# normflag = False
normflag = True
if normflag:
    m_s['locking'] = m_s['locking']/m_s['locking'].max()
    print(m_s['locking'].min(), m_s['locking'].max())



In [ ]:
# plot the inferred slip on the fault

# Inferred slip on the fault interface
# Initialize figure and axis
fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(loc_f['xf']/1e3, loc_f['yf']/1e3, c=m_s['mag'], cmap='hot_r', edgecolors='none', marker='o', 
                s=30, vmin=0, vmax=m_s['mag'].max())
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Magnitude (m)")
ax.set_xlabel("X (km)")
ax.set_ylabel("Y (km)")
ax.set_title("Inferred slip on the fault interface")
ax.set_aspect('equal')

plt.show()

In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -85.5+360, 10
R = 6371.0  # Earth radius in km

# Compute the inverse projection
loc_f['lon'], loc_f['lat'] = ut.inverse_azi_equidist_proj(loc_f['xf']/1e3, loc_f['yf']/1e3, lon0, lat0)
print(loc_f[['lon', 'lat']].head())

df_plate['x'], df_plate['y'] = ut.azi_equidist_proj(df_plate['lon'], df_plate['lat'], lon0, lat0)


In [ ]:
region=[-87+360, -84+360, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 

# Filter by both longitude and latitude
segments = ut.parse_trench_file(datadir + trench_file, 
                           lon_range=(region[0]-360, region[1]-360), 
                           lat_range=(region[2], region[3]))
# Convert to DataFrame for analysis
trench = ut.segments_to_dataframe(segments)

In [ ]:
# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/"
os.makedirs(figpath, exist_ok=True)

In [ ]:
fig = pygmt.Figure()
pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

fig.basemap(region=region, projection="M9c", frame=["a1f0.5", "WSne+tPredicted slip rate"])
# fig.coast(region=region, projection="M10c", frame=["af", "+tInverted slip"], 
#           borders=1, shorelines="0.25p,black", area_thresh=4000) #frame="af", 
pygmt.makecpt(cmap="hot", series=[0, m_s['mag'].max(), m_s['mag'].max()/20], background=True, reverse=True)  #m_s['mag'].max()
# grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['mag'], region=region, spacing=(0.05, 0.05)) # no smoothing
# By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
# data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s['mag'], region=region, spacing=(0.02, 0.02))
# grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.02, 0.02))
# fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation
# fig.grdcontour(grid=grid, annotation=0.02, limit=[0.02, 0.12])
fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s['mag'], cmap=True)   # no smoothing or interpolation

scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.5p,darkgray") 
fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
# for dep in depths:
#   # print(dep)
#   plate_dep = df_plate.loc[df_plate['dep'] == dep]
#   plate_dep = plate_dep.sort_values(by='lat').reset_index(drop=True) 
#   fig.plot(x=plate_dep['lon'], y=plate_dep['lat'], pen="1p,darkgray,--")
fig.colorbar(frame=["af", "x+lMagnitude", "y+l(mm/yr)"])
fig.show()

if flag_savefig:
    fig.savefig(figpath + "slip.pdf")


In [ ]:
fig = pygmt.Figure()
pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

fig.basemap(region=region, projection="M9c", frame=["a1f0.5", "WSne+tPredicted locking"])
# fig.coast(region=region, projection="M10c", frame=["af", "+tInverted slip"], 
#           borders=1, shorelines="0.25p,black", area_thresh=4000) #frame="af", 
# pygmt.makecpt(cmap="hot", series=[0, m_s['locking'].max(), m_s['locking'].max()/20], background=True, reverse=True)  #m_s['mag'].max()
pygmt.makecpt(cmap="rainbow", series=[0, 1, 0.05], background=True, reverse=False)  #m_s['mag'].max()
grid = pygmt.xyz2grd(x=loc_f['lon'], y=loc_f['lat'], z=m_s['locking'], region=region, spacing=(0.04, 0.04)) # no smoothing
# By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
# data_bmedian = pygmt.blockmedian(x=loc_f['lon'], y=loc_f['lat'], z=m_s['locking'], region=region, spacing=(0.02, 0.02))
# grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.02, 0.02))
# fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation
# fig.grdcontour(grid=grid, annotation=0.02, limit=[0.02, 0.12])
fig.plot(x=loc_f['lon'], y=loc_f['lat'], style="c0.125c", fill=m_s['locking'], cmap=True)   # no smoothing or interpolation
fig.grdcontour(grid=grid, limit=[0.5], annotation="0.5+f8p", pen="1p,black")

# scale_lon, scale_lat = region[1]-0.5, region[-2]+0.5
mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)

fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.5p,darkgray") 
fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
# for dep in depths:
#   # print(dep)
#   plate_dep = df_plate.loc[df_plate['dep'] == dep]
#   plate_dep = plate_dep.sort_values(by='lat').reset_index(drop=True) 
#   fig.plot(x=plate_dep['lon'], y=plate_dep['lat'], pen="1p,darkgray,--")
fig.colorbar(frame=["a0.1f0.05", "x+lInterseismic Coupling"])
fig.show()

if flag_savefig:
    fig.savefig(figpath + "slip.pdf")


In [ ]:
# # Construct the displacement vector
# u_obs['mag'] = np.sqrt(u_obs['ux']**2 + u_obs['uy']**2)  # vector length (magnitude)
# u_obs['angle'] = np.degrees(np.arctan2(u_obs['uy'], u_obs['ux']))  # vector direction

# u_pred['mag'] = np.sqrt(u_pred['ux']**2 + u_pred['uy']**2)
# u_pred['angle'] = np.degrees(np.arctan2(u_pred['uy'], u_pred['ux']))

# df.shape, u_obs.shape, u_pred.shape

# # Stack the values into a 2D NumPy array
# u_obs_vectors = np.column_stack((df['lon'], df['lat'], u_obs['angle'], u_obs['mag']*100))
# u_pred_vectors = np.column_stack((df['lon'], df['lat'], u_pred['angle'], u_pred['mag']*100))
# print(u_obs_vectors[:5])

In [ ]:
# A different way of constructing the vectors for plotting arrows
# convert from m to mm, horizontal components
df_obs_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_obs['ux']*1000,
        "north_velocity": u_obs['uy']*1000,
        # "east_sigma": df['sig_ew']*100,
        # "north_sigma": df['sig_ns']*100,
        "east_sigma": error_e*1000,
        "north_sigma": error_n*1000,
        "correlation_EN": 0.0,
    } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
)
df_obs_h.head()

np.ptp(np.sqrt(df_obs_h['east_velocity']**2 + df_obs_h['north_velocity']**2))

# vertical components
df_obs_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_obs['uz']*1000,
        # "east_sigma": df['sig_ew']*100,
        # "north_sigma": df['sig_ns']*100,
        "east_sigma": error_v*1000,
        "north_sigma": error_v*1000,
        "correlation_EN": 0.0,
    } #"SITE": ["0x0", "3x3", "4x6", "6x4", "-6x4", "6x-4"],0.5*
)


In [ ]:
# convert from m to mm
df_pred_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_pred['ux']*1000,
        "north_velocity": u_pred['uy']*1000,        
    }
)
df_pred_h.head()

df_pred_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_pred['uz']*1000,        
    }
)


In [ ]:
# convert from m to mm, in order to directly compare with 
df_res_h = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": u_res['ux']*1000,
        "north_velocity": u_res['uy']*1000,
        "east_sigma": error_e*1000,
        "north_sigma": error_n*1000,
        "correlation_EN": 0.0,        
    }
)
df_res_h.head()

df_res_v = pd.DataFrame(
    data={
        "x": df['lon'],
        "y": df['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_res['uz']*1000,   
        "east_sigma": error_v*1000,
        "north_sigma": error_v*1000,
        "correlation_EN": 0.0,     
    }
)

In [ ]:
# plot the horizontal predicted displacements vs. observed at GNSS stations, and the residual

fig = pygmt.Figure()
with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        # scale_lon, scale_lat = region[0]+1, region[-1]-1
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_h, pen="0.5p,red", uncertaintyfill=None, line="0.25p,red", spec="e0.05/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_h, pen="0.5p,black", line=True, spec="e0.05/0.39", vector="0.1c+a60+p1p+ea+gblack")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.4, "east_velocity": [1], "north_velocity": [0]})
        fig.velo(data=lgdpred, pen="0.5p,black", line=True, spec="e0.5/0.39", vector="0.1c+a60+p1p+ea+gblack")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    # Residual  
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p", spec="e0.1/0.39", 
                 vector="0.1c+a60+p1p+ea+gblack")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.6, "east_velocity": [1], "north_velocity": [0],
                              "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
        fig.velo(data=lgdres, pen="0.5p,black", line="0.25p", spec="e0.5/0.39", vector="0.1c+a60+p1p+ea+gblack")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs-Pred", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="5±1 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")


# fig.plot(data=u_obs_vectors, style="v0.4c+e", pen="2p,black", fill="black")
# fig.plot(data=u_pred_vectors, style="v0.4c+e", pen="1p,red", fill="red") 

fig.show()

if flag_savefig:
    fig.savefig(figpath + "data_misfit_h.pdf")

In [ ]:
# plot the vertical predicted displacements vs. observed at GNSS stations, and the residual

fig = pygmt.Figure()
with fig.subplot(nrows=1, ncols=2, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.5", "WSne"], 
                 margins=["0.1c", "0.2c"],sharey=True):
    
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="12p,Helvetica-Bold,black", MAP_TITLE_OFFSET="0c", MAP_SCALE_HEIGHT="3p")

    # Observed vs. predicted
    with fig.set_panel(panel=[0, 0]):
        # scale_lon, scale_lat = region[0]+1, region[-1]-1
        mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        #observed displacement
        fig.velo(data=df_obs_v, pen="0.5p,red", uncertaintyfill=None, line="0.25p,red", spec="e0.05/0.39", 
                 vector="0.1c+a60+p1p,red+ea+gred")
        # predicted disp. from inferred slip
        fig.velo(data=df_pred_v, pen="0.5p,black", line=True, spec="e0.05/0.39", vector="0.1c+a60+p1p+ea+gblack")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdobs = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                              "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
        fig.velo(data=lgdobs, pen="0.5p,red", line="0.25p,red", spec="e0.5/0.39", vector="0.1c+a60+p1p,red+ea+gred")
        lgdpred = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+0.275, "east_velocity": [0], "north_velocity": [1]})
        fig.velo(data=lgdpred, pen="0.5p,black", line=True, spec="e0.5/0.39", vector="0.1c+a60+p1p+ea+gblack")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Pred", x=region[0]+0.6, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")
        fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

    # Residual  
    with fig.set_panel(panel=[0, 1]):
        fig.coast(region=region, projection="M?", frame="a1f0.5", borders=None, shorelines="0.25p,black",
                  area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
        fig.grdcontour(grid=grd_file, levels=10, limit="-200/0", annotation="20+f8p", pen="0.75p,darkgray") 
        fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
        fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
        # disp. residual == observed - predicted
        fig.velo(data=df_res_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p", spec="e0.05/0.39", 
                 vector="0.1c+a60+p1p+ea+gblack")
        # plot legends
        fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
        fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
        lgdres = pd.DataFrame(data={"x": region[0]+0.3, "y": region[-2]+0.5, "east_velocity": [0], "north_velocity": [1],
                              "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
        fig.velo(data=lgdres, pen="0.5p,black", line="0.25p", spec="e0.5/0.39", vector="0.1c+a60+p1p+ea+gblack")
        # plot texts
        fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
        fig.text(text="Obs-Pred", x=region[0]+0.6, y=region[-2]+0.6, font="8p,Helvetica,black", justify="ML")
        fig.text(text="10±2 mm", x=region[0]+0.1, y=region[-2]+0.4, font="8p,Helvetica,black", justify="ML")


# fig.plot(data=u_obs_vectors, style="v0.4c+e", pen="2p,black", fill="black")
# fig.plot(data=u_pred_vectors, style="v0.4c+e", pen="1p,red", fill="red") 

fig.show()

if flag_savefig:
    fig.savefig(figpath + "data_misfit_v.pdf")

In [ ]:
# plot the histogram of residuals
fig, ax = plt.subplots(figsize=(4, 5), dpi=300)
bin_width = 0.5
xmin, xmax = 0, 10 
bins = np.arange(xmin, xmax + bin_width, bin_width)
counts1, _, _ = ax.hist(u_res['mag']*1000, bins=bins, alpha=0.7, edgecolor='black',label='This study')#, color='blue'

ymin, ymax = 0, np.max(counts1) + 2

median = np.median(u_res['mag']*1000)
sigma = np.std(u_res['mag']*1000)
ax.plot([median, median], [ymin, ymax], '--', color='k', label=fr'median={median:.2f}')
ax.errorbar(median, ymax-0.5, xerr=sigma, fmt='o', color='k', capsize=5, capthick=2, 
            linewidth=2, label=fr'$\pm1\sigma=\pm{sigma:.2f}$')

ax.legend(fontsize=9, loc='upper right')
ax.set_xlabel('Residual (mm)')
ax.set_ylabel('Frequency')
ax.set_title('Data residual (Obs - Pred)', fontweight='bold')
ax.set_ylim(ymin, ymax)
ax.set_yticks(np.arange(ymin, ymax+2, 2))
ax.set_xlim(xmin, xmax)

print(u_res['mag'].max()*1000)

if flag_savefig:
    fig.savefig(figpath + "residual.pdf")


def rmse_3d_dataframe(predicted_df, true_df):
    """Per-component (scalar) RMSE: sqrt(sum(rx^2+ry^2+rz^2)/(3N))."""
    predicted = predicted_df[['ux', 'uy', 'uz']].values
    true = true_df[['ux', 'uy', 'uz']].values
    return np.sqrt(np.mean((predicted - true)**2))

def rmse_3d_vec_dataframe(predicted_df, true_df):
    """Per-vector RMSE: sqrt(sum(|r_i|^2)/N), same units as input vector."""
    predicted = predicted_df[['ux', 'uy', 'uz']].values
    true = true_df[['ux', 'uy', 'uz']].values
    diff = predicted - true                       # shape (N, 3)
    return np.sqrt(np.mean(np.sum(diff**2, axis=1)))

# compute the per-vector RMSE of residual, using: sqrt( Σ|r|² / N ) = sqrt(mean(np.sum(diff**2, axis=1)))
rmse = rmse_3d_vec_dataframe(u_pred, u_obs)* 1000  # Convert to mm
print(f"Residual RMSE: {rmse:.2f} mm")

print(f"Residual median: {median:.2f} mm, sigma: {sigma:.2f} mm")